# 02 · Parcele cu Fields of The World (pe A100)
Descarcă două scene Sentinel-2 (primăvară: aprilie–mai; vară: 1–18 iulie) de pe Planetary Computer pentru fiecare tile MGRS din zonă, rulează modelul FTW și unește poligoanele în `data/out/parcels.parquet` (EPSG:3844).

**Atenție:** zona traversează limita UTM 34/35 la 24° E, deci sunt două tile-uri (34TGP și 35TLJ). Fereastra de vară se oprește pe 18 iulie: după ~23.07.2024 scenele au procesarea 05.11, iar ftw-tools 1.4.3 nu combină versiuni diferite.

In [ ]:
# Sincronizare: aduce repo-ul pe runtime și încarcă configurarea.
# Local (kernel pe PC) găsește direct ../gisday_config.py; pe Colab clonează repo-ul privat.
import os, sys, subprocess, base64, getpass

def _auth(tok):
    b = base64.b64encode(f'x-access-token:{tok}'.encode()).decode()
    return ['-c', f'http.extraheader=AUTHORIZATION: basic {b}']

def _token():
    global TOKEN
    if 'TOKEN' not in globals():
        TOKEN = getpass.getpass('Token GitHub (fine-grained, doar repo-ul gisday2026): ')
    return TOKEN

if not os.path.exists('gisday_config.py'):
    if os.path.exists('../gisday_config.py'):
        os.chdir('..')
    else:
        REPO = 'https://github.com/lefterpatrickandrei-sketch/gisday2026.git'   # același cu GITHUB_REPO din gisday_config.py
        if not os.path.exists('/content/gisday2026'):
            subprocess.run(['git', *_auth(_token()), 'clone', REPO, '/content/gisday2026'], check=True)
        os.chdir('/content/gisday2026')
sys.path.insert(0, os.getcwd())
from gisday_config import *

def pull():
    subprocess.run(['git', *_auth(_token()), 'pull', '--ff-only'], check=True)

def push(paths, msg):
    subprocess.run(['git', 'add', *[str(p) for p in paths]], check=True)
    subprocess.run(['git', '-c', 'user.name=gisday-runtime', '-c', 'user.email=gisday@users.noreply.github.com',
                    'commit', '-m', msg], check=False)
    subprocess.run(['git', *_auth(_token()), 'push'], check=True)

print('Lucrez în', os.getcwd())

In [ ]:
# ftw-tools cere Python >=3.11,<3.13: rulează mereu dintr-un venv separat cu Python 3.12 (uv), oricare ar fi kernelul
v = sys.version_info
print(f'Python kernel {v.major}.{v.minor} (ftw rulează separat, cu Python 3.12)')
ON_COLAB = os.path.exists('/content')
FTW_ENV = '/content/ftwenv' if ON_COLAB else os.path.join(os.getcwd(), '.venv-ftw')
FTW = f'{FTW_ENV}/bin/ftw' if ON_COLAB else os.path.join(FTW_ENV, 'Scripts', 'ftw.exe')

In [ ]:
# Mediul FTW (o dată pe runtime). Versiuni fixate = combinația testată pe 06.11.2026 (--help + download/run/polygonize):
# ftw-tools 1.4.3 importă torchgeo.transforms.AugmentationSequential, care lipsește din torchgeo >= 0.8.
FTW_PKGS = ['ftw-tools==1.4.3', 'torchgeo==0.7.2']
if not os.path.exists(FTW):
    if not ON_COLAB:
        raise RuntimeError('Pe PC: uv venv --python 3.12 .venv-ftw && uv pip install --python .venv-ftw/Scripts/python.exe '
                           + ' '.join(FTW_PKGS))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--clear', '--python', '3.12', FTW_ENV], check=True)  # --clear: reia un venv rămas incomplet
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '-q', '--python', f'{FTW_ENV}/bin/python', *FTW_PKGS], check=True)
print('ftw:', FTW)

In [ ]:
%pip install -q pystac-client planetary-computer geopandas pyarrow shapely earthengine-api

In [ ]:
# Verifică opțiunile reale ale CLI-ului înainte de rulare (nu ghicim flag-urile)
for sub in (['--help'], ['inference', 'download', '--help'], ['inference', 'run', '--help'],
            ['inference', 'polygonize', '--help'], ['model', 'download', '--help']):
    r = subprocess.run([FTW, *sub], capture_output=True, text=True)
    print('$ ftw', ' '.join(sub)); print(r.stdout[-1500:] or r.stderr[-1500:]); print('-' * 60)

In [ ]:
CKPT = INTERIM / '3_Class_FULL_FTW_Pretrained.ckpt'   # THREE_CLASS_FULL (ftw_cli/types.py); uz necomercial
if not CKPT.exists():
    subprocess.run([FTW, 'model', 'download', '--type', 'THREE_CLASS_FULL', '--out', str(CKPT)], check=True)
print('checkpoint:', CKPT, f'{CKPT.stat().st_size / 2**20:.0f} MB')

In [ ]:
import pystac_client, planetary_computer
from shapely.geometry import box, shape
cat = pystac_client.Client.open('https://planetarycomputer.microsoft.com/api/stac/v1',
                                modifier=planetary_computer.sign_inplace)
aoi_ll = box(*AOI_BBOX)

def best(window, max_cloud=10):
    items = cat.search(collections=['sentinel-2-l2a'], bbox=AOI_BBOX, datetime=window,
                       query={'eo:cloud_cover': {'lt': max_cloud}}).item_collection()
    pick = {}
    for it in items:
        t = it.properties.get('s2:mgrs_tile')
        score = (round(shape(it.geometry).intersection(aoi_ll).area, 4), -it.properties['eo:cloud_cover'])
        if t not in pick or score > pick[t][0]:
            pick[t] = (score, it)
    return {t: p[1] for t, p in pick.items()}

# Fereastra B se oprește pe 18 iulie: din ~23.07.2024 Sentinel-2 trece la procesarea 05.11, iar ftw-tools 1.4.3
# refuză perechile cu versiuni diferite (A din aprilie–mai e 05.10). Decizie confirmată de Patrick pe 06.11.2026.
A = best(f'{YEAR}-04-01/{YEAR}-05-31')
B = best(f'{YEAR}-07-01/{YEAR}-07-18')
TILES = sorted(set(A) & set(B))
for t in TILES:
    pa, pb = (x.properties.get('s2:processing_baseline') for x in (A[t], B[t]))
    print(t, A[t].id, A[t].properties['eo:cloud_cover'], pa, '|', B[t].id, B[t].properties['eo:cloud_cover'], pb)
    assert pa == pb, f'{t}: versiuni de procesare diferite ({pa} vs {pb}) — ftw ar refuza perechea'
if set(A) ^ set(B):
    print('ATENȚIE: tile-uri cu scenă bună doar într-o fereastră (pot lăsa goluri):', sorted(set(A) ^ set(B)))

In [ ]:
# Pe fiecare tile: stiva A+B -> model pe GPU -> poligoane. Reluabil: sare peste fișierele deja scrise.
outputs, boxes = [], []
for t in TILES:
    a, b = A[t], B[t]
    inter = shape(a.geometry).intersection(shape(b.geometry)).intersection(aoi_ll)
    if inter.is_empty or inter.area < 1e-4:
        print(t, 'sărit: intersecție prea mică cu zona')
        continue
    boxes.append(box(*inter.bounds))
    bb = ','.join(f'{c:.5f}' for c in inter.bounds)        # lon/lat, cum îl primește odc.stac.load în ftw
    tif, pred, poly = (INTERIM / f'ftw_in_{t}.tif', INTERIM / f'ftw_pred_{t}.tif', INTERIM / f'ftw_poly_{t}.parquet')
    if not tif.exists():
        r = subprocess.run([FTW, 'inference', 'download', '--win_a', a.id, '--win_b', b.id, '--out', str(tif), '--bbox', bb],
                           capture_output=True, text=True)
        print(r.stdout[-600:], r.stderr[-600:])
        # ftw iese cu cod 0 fără fișier dacă scenele au versiuni de procesare diferite sau nu se intersectează
        assert tif.exists(), f'{t}: ftw nu a scris {tif.name} (vezi mesajul de mai sus)'
    if not pred.exists():
        subprocess.run([FTW, 'inference', 'run', str(tif), '--model', str(CKPT), '--out', str(pred), '--gpu', '0'], check=True)
    if not poly.exists():
        subprocess.run([FTW, 'inference', 'polygonize', str(pred), '--simplify', '20', '--out', str(poly)], check=True)
    outputs.append(poly)
    print(t, '->', poly.name)

In [ ]:
import geopandas as gpd, pandas as pd
parts = []
for p in outputs:
    g = gpd.read_parquet(p) if p.suffix == '.parquet' else gpd.read_file(p)
    parts.append(g.to_crs(CRS)[['geometry']])
P = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=CRS)
P = P[P.geometry.is_valid & ~P.geometry.is_empty]
# dubluri la suprapunerea tile-urilor: păstrează poligonul mai mare când suprapunerea depășește 50% din cel mai mic
# (raportarea la cel mai mic prinde și dublurile tăiate la marginea unui tile, pe care IoU le-ar scăpa)
P['a'] = P.area
drop = set()
for i, j in zip(*P.sindex.query(P.geometry, predicate='intersects')):
    if i < j and i not in drop and j not in drop:
        gi, gj = P.geometry.iloc[i], P.geometry.iloc[j]
        if gi.intersection(gj).area / min(gi.area, gj.area) > 0.5:
            drop.add(j if P.a.iloc[i] >= P.a.iloc[j] else i)
P = P.drop(P.index[list(drop)])

In [ ]:
import ee
try:
    ee.Initialize(project=GEE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='notebook')   # afișează un link; lipești codul primit
    ee.Initialize(project=GEE_PROJECT)
BBOX = ee.Geometry.Rectangle(list(AOI_BBOX))
RO = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'Romania'))
AOI = BBOX.intersection(RO.geometry(), ee.ErrorMargin(10))
print('Earth Engine OK')

In [ ]:
from shapely.geometry import shape as _shape
aoi_poly = gpd.GeoSeries([_shape(AOI.getInfo())], crs='EPSG:4326').to_crs(CRS).iloc[0]
P = P[P.intersects(aoi_poly)].copy()
P['geometry'] = P.intersection(aoi_poly)
P['area_ha'] = P.area / 1e4
P = P[P.area_ha >= 0.5].reset_index(drop=True)
P['parcel_id'] = range(1, len(P) + 1)
P[['parcel_id', 'area_ha', 'geometry']].to_parquet(OUT / 'parcels.parquet')
print(len(P), 'parcele;', round(P.area_ha.sum()), 'ha')

In [ ]:
# Verificare: număr de parcele, acoperirea zonei, continuitatea la limita UTM 34/35 (24° E)
import numpy as np
aoi_km2 = aoi_poly.area / 1e6
proc = gpd.GeoSeries(boxes, crs=4326).to_crs(CRS).union_all().intersection(aoi_poly)
print(f'{len(P)} parcele | {P.area_ha.sum() / 100:.0f} km² parcele din {aoi_km2:.1f} km² AOI ({P.area_ha.sum() / 100 / aoi_km2:.0%})')
print(f'zona procesată de FTW: {proc.area / aoi_poly.area:.1%} din AOI (neacoperit: {(aoi_poly.area - proc.area) / 1e6:.1f} km²)')
rows = []
for lon in np.round(np.arange(23.90, 24.10, 0.01), 2):       # fâșii de ~0.8 km de o parte și de alta a 24° E
    s = gpd.GeoSeries([box(lon, AOI_BBOX[1], lon + 0.01, AOI_BBOX[3])], crs=4326).to_crs(CRS).iloc[0].intersection(aoi_poly)
    rows.append((lon, P.intersection(s).area.sum() / s.area if s.area else np.nan))
prof = pd.DataFrame(rows, columns=['lon', 'frac_parcele'])
print(prof.round(3).to_string(index=False))
gol = prof[prof.lon.between(23.98, 24.01) & (prof.frac_parcele < 0.5 * prof.frac_parcele.median())]
print('GOL la 24° E:' if len(gol) else 'Fără gol la 24° E (fâșiile 23.98–24.01 au cel puțin jumătate din mediana vecinilor)', gol.lon.tolist() if len(gol) else '')

In [ ]:
push([OUT / 'parcels.parquet'], 'parcele FTW')